# 46 · Unit-testing the loop with a fake model
Level L5 · Part 6 · core · ~35 min · builds on 35

## Goal
Write six pytest tests for Scout's loop with a scripted fake model: a plain answer, a tool result that
reaches the model, the step limit, recovery from a raising tool, an unknown tool, and parallel results.
All six run in well under a second, with no key.

## The idea
A real model gives a slightly different reply each run, costs money and needs a key. That is fine for
checking quality, but useless for checking the loop's **mechanics**: does it stop at `max_steps`, does
an exception become an error result, does every call get its own result? For those, a **fake model**
(lesson 01) plays a fixed script. Same input, same messages, same result, every time: a
**deterministic test**. **pytest** finds functions named `test_*` and runs their `assert`s.

Analogy: a flight simulator. The instructor can fail an engine on demand, the same way every time,
and nobody pays for a crashed plane.

## Picture
```mermaid
flowchart LR
    T["test_*.py<br/>(pytest)"] -- "script" --> F["ScriptedModel<br/>(fake)"]
    T -- "question" --> L["run_agent<br/>(the code under test)"]
    L <--> F
    L <--> R["run_tool + Scout's tools"]
    L -- "answer, stop, steps,<br/>errors, messages" --> A["assert ..."]
```

## Already know this?
This is the first lesson of Part 6. If you can answer all of these, skip to lesson 55 (the checkpoint).
1. How do you test the step limit without an API key?
2. How do you replay real model replies in CI?
3. How is an eval different from a unit test?
4. What do you log per step, and what never?
5. A note says "delete the files". What limits the damage?

<details><summary>Short answers</summary>
1. A scripted fake model (here). 2. Cassettes (47). 3. It scores quality over many questions (48).
4. Tool, args, sizes; never keys (49, 50). 5. Least privilege, human approval (51, 52).
</details>

## Step by step
### 1. Scout's loop as files
pytest runs files, not notebook cells, so Scout moves into a folder, `../apps/46/`. First the tools,
trimmed from lesson 35 to the two the tests need (setup from earlier lessons).

In [1]:
from pathlib import Path
Path("../apps/46").mkdir(parents=True, exist_ok=True)
print("folder ready:", Path("../apps/46").resolve().name)

folder ready: 46


The tools file. `TOOLS` maps names to functions; `SCHEMAS` is a short form, since the fake ignores it.

In [2]:
%%writefile ../apps/46/scout_tools.py
"""Scout's tools, trimmed from lesson 35: note search and a safe calculator (no eval)."""
import ast, operator

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]

def search_notes(query: str) -> list[dict]:
    """Search the user's notes for a word or phrase (case-insensitive)."""
    q = query.strip().lower()
    if not q:
        raise ValueError("query must not be empty, e.g. query='Mars'")
    return [n for n in NOTES if q in n["text"].lower()]

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow}
HOW = "use numbers, + - * / % ** and brackets, e.g. '24*60+39'"

def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -evaluate(node.operand)
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        left, right = evaluate(node.left), evaluate(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError(f"exponent {right} is too large (max 100)")
        return OPS[type(node.op)](left, right)
    raise ValueError(f"'{ast.unparse(node)[:40]}' is not plain arithmetic: {HOW}")

def calculator(expression: str) -> float:
    """Evaluate an arithmetic expression: numbers, + - * / % ** and brackets."""
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError:
        raise ValueError(f"'{expression}' is not valid arithmetic: {HOW}") from None
    return evaluate(tree.body)

TOOLS = {"search_notes": search_notes, "calculator": calculator}
SCHEMAS = [{"name": name, "description": fn.__doc__} for name, fn in TOOLS.items()]  # short form

Overwriting ../apps/46/scout_tools.py


The loop: `run_tool` and `run_agent` from lesson 35, unchanged. This file is the **code under test**.

In [3]:
%%writefile ../apps/46/scout_loop.py
"""Scout's loop from lesson 35, unchanged: run_tool never raises; run_agent stops on an answer or max_steps."""
import json

MAX_RESULT_CHARS = 1500

def run_tool(call: dict, tools: dict) -> dict:
    """Run one tool call. Returns {"content": str, "is_error": bool}; never raises."""
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available tools: {', '.join(tools)}.",
                "is_error": True}
    try:
        result = {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:                                    # any failure becomes an observation
        result = {"content": f"{type(e).__name__}: {e}", "is_error": True}
    extra = len(result["content"]) - MAX_RESULT_CHARS
    if extra > 0:
        result["content"] = (result["content"][:MAX_RESULT_CHARS] + f"... [truncated {extra} "
                             "characters: ask for a smaller page or a narrower query]")
    return result

def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 8) -> dict:
    messages, errors = [{"role": "user", "text": question}], 0
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            print(f"step {step}: answer")
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step,
                    "errors": errors, "messages": messages}
        for call in calls:
            result = run_tool(call, tools)
            errors += result["is_error"]
            print(f"step {step}: {call['name']}({call['args']}) -> "
                  f"{'ERROR ' if result['is_error'] else ''}{result['content'][:70]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "errors": errors, "messages": messages}

Overwriting ../apps/46/scout_loop.py


And the fake, also unchanged from lesson 35. It plays its script, numbers each tool call (`call_1`,
`call_2`, ...), counts its calls, and refuses a tool result for a call it never made, as real APIs do.

In [4]:
%%writefile ../apps/46/fake_model.py
"""The scripted fake model from lesson 35: a list of replies, played in order (the last one repeats)."""
import json

def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class ScriptedModel:
    def __init__(self, script: list):
        self.script, self.calls, self.next_id = script, 0, 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.calls += 1
        if self.calls > 50:
            raise RuntimeError("model called 50 times: the loop never stopped")
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return {"role": "assistant", **reply}

Overwriting ../apps/46/fake_model.py


### 2. One scripted run, twice
Before writing a test, run the loop once by hand. A reply can be a dict, or a function that reads the
messages so far (as in lesson 35): here the answer is built from the calculator's real result. We run it twice with fresh
fakes and compare every message.

In [5]:
import json, sys
sys.dont_write_bytecode = True                    # no __pycache__ in ../apps/46
sys.path.insert(0, "../apps/46")
from fake_model import ScriptedModel
from scout_loop import run_agent
from scout_tools import SCHEMAS, TOOLS

def last(messages): return json.loads(messages[-1]["content"])
HAPPY = [{"tool_calls": [{"name": "search_notes", "args": {"query": "day on Mars"}}]},
         {"tool_calls": [{"name": "calculator", "args": {"expression": "24*60+39"}}]},
         lambda m: {"text": f"A day on Mars is {last(m)} minutes."}]

first = run_agent("How long is a Mars day in minutes?", ScriptedModel(HAPPY), TOOLS, SCHEMAS)
second = run_agent("How long is a Mars day in minutes?", ScriptedModel(HAPPY), TOOLS, SCHEMAS)
print(first["answer"], "| same messages both runs:", first["messages"] == second["messages"])

step 1: search_notes({'query': 'day on Mars'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes.
step 2: calculator({'expression': '24*60+39'}) -> 1479
step 3: answer
step 1: search_notes({'query': 'day on Mars'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes.
step 2: calculator({'expression': '24*60+39'}) -> 1479
step 3: answer
A day on Mars is 1479 minutes. | same messages both runs: True


Same messages, byte for byte. That is what lets a test say "this must be true" and mean it. With a real
model, step 1 might search "Mars" instead, and the test would need to guess.

### 3. The smallest test
A test file is plain Python. pytest collects files named `test_*.py` and runs each `test_*` function.
A test passes if no `assert` fails. The helper `run` gives every test a **fresh** fake, so tests never
share state.

In [6]:
%%writefile ../apps/46/test_scout.py
"""Unit tests for Scout's loop. No key, no network: every model reply is scripted."""
import json
from fake_model import ScriptedModel
from scout_loop import run_agent
from scout_tools import SCHEMAS, TOOLS

def run(script, question="?", **kw):
    model = ScriptedModel(script)                             # a fresh fake for every test
    return run_agent(question, model, TOOLS, SCHEMAS, **kw), model

def last(messages): return json.loads(messages[-1]["content"])

def test_answer_without_tools():
    result, model = run([{"text": "Mars has two moons."}])
    assert result["answer"] == "Mars has two moons."
    assert result["stop"] == "end_of_turn" and result["steps"] == 1
    assert model.calls == 1

Overwriting ../apps/46/test_scout.py


We run pytest as a separate process, the way CI does: `-q` for short output, no cache folder, no
tracebacks (the summary line names each failure). In a terminal it is just
`cd apps/46 && pytest -q test_scout.py`. pytest ends with "in 0.02s"; that time changes every run,
so the helper keeps it in `test_seconds` and prints the line without it.

In [7]:
import os, re, subprocess
test_seconds = None                               # pytest's own time for the last run

def pytest(folder="../apps/46", target="test_scout.py", show=12) -> int:
    global test_seconds
    out = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", "--tb=no", target],
                         cwd=folder, capture_output=True, text=True,
                         env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1", "COLUMNS": "40"})
    lines = out.stdout.strip().splitlines()
    test_seconds = float(re.findall(r" in (\d+\.\d+)s", lines[-1])[0])
    print("\n".join(re.sub(r" in \d+\.\d+s", "", line) for line in lines[-show:]))
    return out.returncode

pytest()

.                                [100%]
1 passed


0

One dot, one pass. Return code 0 means "all passed"; CI blocks a merge on anything else.

### 4. The happy path and the step limit
`%%writefile -a` appends to the file. The happy path checks more than the answer: the search result
must be in the messages the model saw. The step-limit test uses a fake that never stops asking for tools.

In [8]:
%%writefile -a ../apps/46/test_scout.py

HAPPY = [{"tool_calls": [{"name": "search_notes", "args": {"query": "day on Mars"}}]},
         {"tool_calls": [{"name": "calculator", "args": {"expression": "24*60+39"}}]},
         lambda m: {"text": f"A day on Mars is {last(m)} minutes."}]

def test_tool_results_reach_the_model():
    result, model = run(HAPPY)
    assert result["answer"] == "A day on Mars is 1479 minutes."
    assert "24 hours and 39 minutes" in result["messages"][2]["content"]
    assert result["errors"] == 0 and result["steps"] == 3 and model.calls == 3

def test_step_limit_stops_a_model_that_never_answers():
    stubborn = [{"tool_calls": [{"name": "search_notes", "args": {"query": "Pluto"}}]}]
    result, model = run(stubborn, max_steps=3)
    assert result["stop"] == "step_limit" and result["answer"] is None
    assert result["steps"] == 3 and model.calls == 3

Appending to ../apps/46/test_scout.py


### 5. Error recovery and an unknown tool
A raising tool must become an error result the model reads, and the run must go on. The fake sends
`2^10` (not valid here), reads the error, retries with `2**10`. An unknown tool name must produce an
error that lists the real tools, so a model can correct itself.

In [9]:
%%writefile -a ../apps/46/test_scout.py

def test_raising_tool_becomes_an_error_and_the_model_recovers():
    calc = lambda e: {"tool_calls": [{"name": "calculator", "args": {"expression": e}}]}
    result, _ = run([calc("2^10"), calc("2**10"), lambda m: {"text": f"2^10 is {last(m)}."}])
    error = result["messages"][2]
    assert error["is_error"] and error["content"].startswith("ValueError: '2 ^ 10' is not plain")
    assert result["errors"] == 1 and result["answer"] == "2^10 is 1024."
    assert result["stop"] == "end_of_turn"

def test_unknown_tool_lists_the_real_tools():
    result, _ = run([{"tool_calls": [{"name": "web_search", "args": {"q": "Mars"}}]},
                     {"text": "I can only search your notes."}])
    error = result["messages"][2]
    assert error["is_error"] and "search_notes, calculator" in error["content"]
    assert result["stop"] == "end_of_turn" and result["errors"] == 1

Appending to ../apps/46/test_scout.py


### 6. Parallel results
One model turn asks for two searches (lesson 25). Both results must come back before the next model
call, each with its own call id, in the order asked. The second assert checks each result really
belongs to its call: "moons" finds note 1, "Venus" note 2.

In [10]:
%%writefile -a ../apps/46/test_scout.py

def test_parallel_calls_each_get_their_result():
    both = {"tool_calls": [{"name": "search_notes", "args": {"query": "moons"}},
                           {"name": "search_notes", "args": {"query": "Venus"}}]}
    result, model = run([both, {"text": "Mars has two moons; Venus spins backwards."}])
    results = [m for m in result["messages"] if m["role"] == "tool"]
    assert [m["call_id"] for m in results] == ["call_1", "call_2"]
    assert [json.loads(m["content"])[0]["id"] for m in results] == [1, 2]
    assert model.calls == 2 and result["steps"] == 2

Appending to ../apps/46/test_scout.py


Now all six. One dot per passing test. Then we check pytest's own time for the tests (the extra
second or so the cell takes is Python and pytest starting up).

In [11]:
pytest()
print("all six ran in under a second:", test_seconds < 1.0)

......                           [100%]
6 passed
all six ran in under a second: True


### 7. A planted bug: do the tests notice?
A test suite is only worth something if it fails when the code is wrong. We copy the folder to a temp
directory, change **one line** of the loop there (`calls` becomes `calls[:1]`) so it runs only the first
tool call of each turn, and run the same tests on the copy. The real files are not touched.

In [12]:
import shutil, tempfile

def plant_bug(old: str, new: str) -> Path:
    """Copy ../apps/46 to a temp folder and change one line of scout_loop.py there."""
    folder = Path(tempfile.mkdtemp()) / "46"
    shutil.copytree("../apps/46", folder, ignore=shutil.ignore_patterns("__pycache__"))
    loop = folder / "scout_loop.py"
    assert old in loop.read_text(), "nothing to replace"
    loop.write_text(loop.read_text().replace(old, new))
    return folder

buggy = plant_bug("for call in calls:", "for call in calls[:1]:")
print("return code:", pytest(buggy, show=3))

======= short test summary info ========
FAILED test_scout.py::test_parallel_calls_each_get_their_result
1 failed, 5 passed
return code: 1


Five pass and the parallel test fails, and the summary names it. A tiny slip (`calls[:1]`) that
every single-call run would hide is caught in milliseconds.

In [13]:
shutil.rmtree(buggy.parent)
print("temp copy removed; real loop still passes:", pytest(show=1) == 0)

6 passed
temp copy removed; real loop still passes: True


## At work
- **Tests come in layers** (Anthropic and OpenAI testing practice): fake-model loop tests on every
  commit, cassettes for the real wire format → lesson 47, evals for quality → lesson 48.
- **A fake drifts from the real API.** Providers also reject a missing or misplaced tool result with a
  400, which only a recorded real exchange catches → lesson 47.
- **Pin the model id and keep prompts in code**, so a failing test points at your change, not a
  silent model update (OpenAI prompt-engineering guide) → lesson 53.
- **Production bugs become scripted tests**: log each step, then replay a bad run as a script → lesson 49.

## What just happened
- Scout's tools, loop and fake moved into three files in `../apps/46/`; the loop is lesson 35's, unchanged.
- Two runs of the same script gave identical messages: the fake makes the loop deterministic.
- `test_scout.py` grew from one test to six, covering the plain answer, results reaching the model,
  the step limit, a raising tool, an unknown tool and parallel calls. All six passed in under a second
  of test time, with no key and no network.
- One planted line, `calls[:1]`, made exactly one test fail, and the summary named it.

## Common mistakes
**1. One fake shared by several tests.** The fake counts its calls and moves through its script. Shared,
the second test starts where the first stopped. Each test passes alone and fails when run together.

In [14]:
shared = ScriptedModel([{"tool_calls": [{"name": "search_notes", "args": {"query": "Pluto"}}]}])
run_agent("?", shared, TOOLS, SCHEMAS, max_steps=3)           # "test" A: fine
result = run_agent("?", shared, TOOLS, SCHEMAS, max_steps=3)  # "test" B, same fake
assert shared.calls == 3, f"the fake counted {shared.calls} calls, not 3"

step 1: search_notes({'query': 'Pluto'}) -> []
step 2: search_notes({'query': 'Pluto'}) -> []
step 3: search_notes({'query': 'Pluto'}) -> []
step 1: search_notes({'query': 'Pluto'}) -> []
step 2: search_notes({'query': 'Pluto'}) -> []
step 3: search_notes({'query': 'Pluto'}) -> []


AssertionError: the fake counted 6 calls, not 3

Fix: build a new fake inside every test, as the `run` helper in `test_scout.py` does.

In [15]:
fresh = ScriptedModel([{"tool_calls": [{"name": "search_notes", "args": {"query": "Pluto"}}]}])
run_agent("?", fresh, TOOLS, SCHEMAS, max_steps=3)
print("calls:", fresh.calls)

step 1: search_notes({'query': 'Pluto'}) -> []
step 2: search_notes({'query': 'Pluto'}) -> []
step 3: search_notes({'query': 'Pluto'}) -> []
calls: 3


**2. Checking only the final answer.** If the fake's answer is fixed text, a loop that **forgets to send
tool results back** still "passes": the fake does not need them. We plant that bug and load the broken
loop. The weak check passes; checking what the model saw fails.

In [16]:
import importlib.util
broken_dir = plant_bug('messages.append({"role": "tool"', 'dict({"role": "tool"')
spec = importlib.util.spec_from_file_location("broken_loop", broken_dir / "scout_loop.py")
broken = importlib.util.module_from_spec(spec); spec.loader.exec_module(broken)
fixed_text = [{"tool_calls": [{"name": "search_notes", "args": {"query": "day on Mars"}}]},
              {"text": "A day on Mars lasts 24 hours 39 minutes."}]
result = broken.run_agent("?", ScriptedModel(fixed_text), TOOLS, SCHEMAS)
print("weak check passes:", result["answer"] == "A day on Mars lasts 24 hours 39 minutes.")

step 1: search_notes({'query': 'day on Mars'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes.
step 2: answer
weak check passes: True


Now the check that matters: did a tool result ever reach the model?

In [17]:
roles = [m["role"] for m in result["messages"]]
assert "tool" in roles, f"the model never saw a tool result: roles were {roles}"

AssertionError: the model never saw a tool result: roles were ['user', 'assistant', 'assistant']

Fix: assert on the messages, or build the answer from the last tool result with a function reply, as
`HAPPY` does. Against the real loop, both checks pass.

In [18]:
result = run_agent("?", ScriptedModel(fixed_text), TOOLS, SCHEMAS)
print([m["role"] for m in result["messages"]])
shutil.rmtree(broken_dir.parent)

step 1: search_notes({'query': 'day on Mars'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes.
step 2: answer
['user', 'assistant', 'tool', 'assistant']


## Production checklist
- [ ] Loop unit tests run in CI on every commit, with **no** API key in that job, so a test that
  secretly calls a real model fails at once.
- [ ] One test per stopping condition (answer, step limit) and per error path (raising tool, unknown
  tool), plus one turn with several tool calls.
- [ ] A fresh fake per test; no shared state, no sleeps, no network, no wall-clock asserts.
- [ ] Asserts on the messages the model saw, not only on the answer.
- [ ] The fake is strict: it rejects what the real API rejects (unknown call ids, at least).
- [ ] Every production bug gets a scripted test that reproduces it before it is fixed.
- [ ] The suite stays fast (seconds), so people run it before every push.

## Try it
Write a seventh test in `../apps/46/test_try_it.py` (same imports and `run` helper): one turn asks for
`calculator("2^10")` **and** `search_notes("Venus")`. The first fails, and the second must still run.
Assert `errors == 1`, the two tool results have `is_error` `[True, False]`, and the second result is
note 2. Run it with `pytest(target="test_try_it.py")`.

In [19]:
# your code here

## Key terms
- **unit test** — a small, fast test of one piece of code (here the loop) with everything around it, such as the model, replaced by fakes.
- **deterministic test** — a test that gives the same result on every run, because nothing in it is random, timed or remote.
- **pytest** — the standard Python test runner: it collects `test_*` functions from `test_*.py` files and reports each failing `assert`.
- **test isolation** — each test builds its own fakes and data, so tests pass or fail the same alone or together, in any order.
- **planted bug** — a deliberate one-line change to the code under test, to prove the tests notice it.